# Explore the movie data

This notebook is for exploratory analysis after the normalized movie tables have passed the data-quality profiling step. It loads the local cache without making API requests.


In [1]:
from IPython.display import display
import pandas as pd

from streamvault.cache import load_cached_records
from streamvault.config import CONFIG
from streamvault.normalize import normalize_movie_records

records = list(load_cached_records(CONFIG.cache_file).values())
tables = normalize_movie_records(records)
movies = tables.movies

frames = {
    name: getattr(tables, name)
    for name in (
        "movies", "genres", "movie_genres", "companies",
        "movie_companies", "countries", "movie_origin_countries",
        "movie_production_countries", "languages",
        "movie_spoken_languages", "collections", "people",
        "cast_credits", "crew_credits",
    )
}

print(f"Loaded {len(records):,} cached movie records")
if not records:
    print("No cached movies yet. Run `uv run streamvault` first.")

# Format to 4 decimal places with comma separators (e.g., 108,446,000.00)
pd.set_option('display.float_format', '{:,.4f}'.format)


Loaded 112 cached movie records


## 3. Numeric movie fields and unusual values

Use quantiles and interquartile-range (IQR) bounds to locate values worth checking. These bounds are exploratory, not universal validity rules; budget and revenue may be zero or absent, for example, and an extreme value may be legitimate.


In [2]:
identifier_columns = {"movie_id", "collection_id"}
numeric_movie_columns = [
    column
    for column in movies.select_dtypes(include="number").columns
    if column not in identifier_columns
]

if numeric_movie_columns:
    numeric_summary = movies[numeric_movie_columns].describe(
        percentiles=[0.01, 0.25, 0.50, 0.75, 0.90, 0.99]
    ).T
    display(numeric_summary)
else:
    print("No numeric movie measures are available.")

outlier_rows = []
for column in numeric_movie_columns:
    values = pd.to_numeric(movies[column], errors="coerce")
    valid_values = values.dropna()
    if valid_values.empty:
        continue

    q1 = valid_values.quantile(0.25)
    q3 = valid_values.quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    unusual = values.lt(lower_bound) | values.gt(upper_bound)
    examples = movies.loc[unusual, ["movie_id", column]].head(5)

    outlier_rows.append(
        {
            "field": column,
            "iqr_lower_bound": lower_bound,
            "iqr_upper_bound": upper_bound,
            "values_outside_iqr": int(unusual.sum()),
            "negative_values": int(valid_values.lt(0).sum()),
            "example_movie_values": [
                f"{row.movie_id}: {row_value}"
                for row, row_value in zip(
                    examples.itertuples(index=False),
                    examples[column],
                    strict=True,
                )
            ],
        }
    )

outlier_review = pd.DataFrame(outlier_rows)
display(outlier_review)


,count,mean,std,min,1%,25%,50%,75%,90%,99%,max
budget,112.0000,"108,445,982.1429","82,788,177.8615","1,200,000.0000","4,665,000.0000","38,087,500.0000","91,000,000.0000","150,000,000.0000","229,000,000.0000","349,840,000.0000","379,000,000.0000"
popularity,112.0000,19.6574,13.4498,0.4583,1.0591,10.3549,16.3064,23.9677,36.8038,65.7337,89.1177
revenue,112.0000,"599,658,765.7411","436,246,692.9360","7,390,108.0000","29,502,907.4100","295,346,340.0000","519,563,017.5000","872,698,393.5000","1,057,280,933.6000","1,993,719,091.3600","2,942,649,778.0000"
runtime,112.0000,130.4911,25.5368,81.0000,81.8800,115.0000,128.5000,143.2500,164.7000,200.3400,202.0000
vote_average,112.0000,7.5887,0.7135,5.7520,6.0359,7.2012,7.7200,8.1645,8.4510,8.6753,8.7340
vote_count,112.0000,"17,538.3214","8,270.0406","4,136.0000","5,959.5000","11,039.0000","15,841.0000","22,821.5000","29,265.2000","40,713.8500","41,925.0000"


,field,iqr_lower_bound,iqr_upper_bound,values_outside_iqr,negative_values,example_movie_values
0,budget,"-129,781,250.0000","317,868,750.0000",2,0,"[299534: 356000000, 1865: 379000000]"
1,popularity,-10.0644,44.3869,6,0,"[155: 66.8926, 278: 45.4678, 299534: 89.1177, ..."
2,revenue,"-570,681,740.2500","1,738,726,473.7500",2,0,"[299534: 2942649778, 299536: 2052415039]"
3,runtime,72.6250,185.6250,3,0,"[122: 201, 240: 202, 424: 195]"
4,vote_average,5.7564,9.6094,1,0,[41513: 5.752]
5,vote_count,"-6,634.7500","40,495.2500",2,0,"[27205: 40754, 157336: 41925]"


#### 3b. Investigating the unusual values
IQR flags are screening candidates, not validity rules. The checks below separate missing or negative values from plausible tail observations, and compare each candidate with related fields before drawing a preliminary conclusion.

In [3]:
investigation_movies = movies.copy()
if "release_date" in investigation_movies.columns:
    investigation_movies["release_year"] = investigation_movies["release_date"].dt.year

unusual_details = []
for column in numeric_movie_columns:
    values = pd.to_numeric(investigation_movies[column], errors="coerce")
    valid_values = values.dropna()
    if valid_values.empty:
        continue

    q1 = valid_values.quantile(0.25)
    q3 = valid_values.quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    unusual = values.lt(lower_bound) | values.gt(upper_bound)

    for row in investigation_movies.loc[unusual].itertuples(index=False):
        value = getattr(row, column)
        unusual_details.append(
            {
                "field": column,
                "movie_id": row.movie_id,
                "title": row.title if "title" in investigation_movies.columns else pd.NA,
                "release_year": (
                    getattr(row, "release_year")
                    if "release_year" in investigation_movies.columns
                    else pd.NA
                ),
                "value": value,
                "tail": "low" if value < lower_bound else "high",
                "iqr_lower_bound": lower_bound,
                "iqr_upper_bound": upper_bound,
            }
        )

unusual_detail_table = pd.DataFrame(unusual_details)
display(unusual_detail_table)

,field,movie_id,title,release_year,value,tail,iqr_lower_bound,iqr_upper_bound
0,budget,299534,Avengers: Endgame,2019,"356,000,000.0000",high,"-129,781,250.0000","317,868,750.0000"
1,budget,1865,Pirates of the Caribbean: On Stranger Tides,2011,"379,000,000.0000",high,"-129,781,250.0000","317,868,750.0000"
2,popularity,155,The Dark Knight,2008,66.8926,high,-10.0644,44.3869
3,popularity,278,The Shawshank Redemption,1994,45.4678,high,-10.0644,44.3869
4,popularity,299534,Avengers: Endgame,2019,89.1177,high,-10.0644,44.3869
5,popularity,27205,Inception,2010,45.4139,high,-10.0644,44.3869
6,popularity,299536,Avengers: Infinity War,2018,49.4655,high,-10.0644,44.3869
7,popularity,157336,Interstellar,2014,56.3569,high,-10.0644,44.3869
8,revenue,299534,Avengers: Endgame,2019,"2,942,649,778.0000",high,"-570,681,740.2500","1,738,726,473.7500"
9,revenue,299536,Avengers: Infinity War,2018,"2,052,415,039.0000",high,"-570,681,740.2500","1,738,726,473.7500"


##### Financial values: budget and revenue
These checks look for contradictory signs or scale errors, not merely large nominal amounts. A large budget or revenue is possible for a blockbuster; a revenue below zero, a negative budget, or a ratio that lacks a sensible explanation would be a stronger anomaly signal.

In [4]:
financial_columns = [column for column in ("budget", "revenue") if column in investigation_movies.columns]
if financial_columns:
    financial = investigation_movies[["movie_id", "title", "release_year", *financial_columns]].copy()
    for column in financial_columns:
        financial[f"{column}_missing"] = financial[column].isna()
        financial[f"{column}_negative"] = financial[column].lt(0)
        financial[f"{column}_zero"] = financial[column].eq(0)

    if "budget" in financial and "revenue" in financial:
        financial["revenue_budget_ratio"] = financial["revenue"] / financial["budget"].replace(0, pd.NA)

    print("Financial integrity checks:")
    for column in financial_columns:
        values = pd.to_numeric(financial[column], errors="coerce")
        print(
            f"{column}: missing={values.isna().sum()}, zero={values.eq(0).sum()}, "
            f"negative={values.lt(0).sum()}"
        )

    financial_extreme_ids = set(
        unusual_detail_table.loc[
            unusual_detail_table["field"].isin(financial_columns), "movie_id"
        ]
    )
    if financial_extreme_ids:
        display(financial.loc[financial["movie_id"].isin(financial_extreme_ids)].sort_values("movie_id").reset_index(drop=True))

    if "revenue_budget_ratio" in financial:
        ratio_summary = financial["revenue_budget_ratio"].describe(percentiles=[0.25, 0.5, 0.75, 0.90, 0.99])
        print("Revenue/budget ratio summary (infinite values excluded):")
        display(ratio_summary.to_frame("ratio").T)

        ratio_candidates = financial.loc[
            financial["revenue_budget_ratio"].notna()
            & (financial["revenue_budget_ratio"] < 0),
            ["movie_id", "title", "release_year", "budget", "revenue", "revenue_budget_ratio"],
        ]
        if not ratio_candidates.empty:
            display(ratio_candidates)
        else:
            print("No negative revenue/budget ratios; no ratio with a negative numerator or denominator was found.")

Financial integrity checks:
budget: missing=0, zero=0, negative=0
revenue: missing=0, zero=0, negative=0


,movie_id,title,release_year,budget,revenue,budget_missing,budget_negative,budget_zero,revenue_missing,revenue_negative,revenue_zero,revenue_budget_ratio
0,1865,Pirates of the Caribbean: On Stranger Tides,2011,379000000,1046721266,False,False,False,False,False,False,2.7618
1,299534,Avengers: Endgame,2019,356000000,2942649778,False,False,False,False,False,False,8.2659
2,299536,Avengers: Infinity War,2018,300000000,2052415039,False,False,False,False,False,False,6.8414


Revenue/budget ratio summary (infinite values excluded):


,count,mean,std,min,25%,50%,75%,90%,99%,max
ratio,112.0000,8.5036,10.3771,1.1337,3.7969,5.3741,8.8502,14.5937,64.3331,70.4907


No negative revenue/budget ratios; no ratio with a negative numerator or denominator was found.


##### Runtime values
The IQR high runtime candidates are reviewed with neighboring runtimes and release context. Runtime units, missing values, and a clearly implausible duration would be stronger anomaly evidence than being above a percentile bound.

In [5]:
if "runtime" in investigation_movies.columns:
    runtimes = pd.to_numeric(investigation_movies["runtime"], errors="coerce")
    print(
        f"runtime: missing={runtimes.isna().sum()}, zero={runtimes.eq(0).sum()}, "
        f"negative={runtimes.lt(0).sum()}"
    )
    runtime_extremes = unusual_detail_table.loc[unusual_detail_table["field"] == "runtime"]
    if not runtime_extremes.empty:
        display(
            investigation_movies.loc[
                investigation_movies["movie_id"].isin(runtime_extremes["movie_id"]),
                ["movie_id", "title", "release_year", "runtime"],
            ].sort_values("runtime", ascending=False).reset_index(drop=True)
        )
    print("Longest runtimes overall:")
    display(
        investigation_movies.sort_values("runtime", ascending=False)[
            ["movie_id", "title", "release_year", "runtime", "vote_average"]
        ].head(12).reset_index(drop=True)
    )

runtime: missing=0, zero=0, negative=0


,movie_id,title,release_year,runtime
0,240,The Godfather Part II,1974,202
1,122,The Lord of the Rings: The Return of the King,2003,201
2,424,Schindler's List,1993,195


Longest runtimes overall:


,movie_id,title,release_year,runtime,vote_average
0,240,The Godfather Part II,1974,202,8.5730
1,122,The Lord of the Rings: The Return of the King,2003,201,8.5070
2,424,Schindler's List,1993,195,8.5700
3,299534,Avengers: Endgame,2019,181,8.2470
4,121,The Lord of the Rings: The Two Towers,2002,179,8.4330
5,120,The Lord of the Rings: The Fellowship of the Ring,2001,179,8.4510
6,238,The Godfather,1972,175,8.6880
7,111,Scarface,1983,170,8.1540
8,49051,The Hobbit: An Unexpected Journey,2012,169,7.3890
9,285,Pirates of the Caribbean: At World's End,2007,169,7.2810


##### Popularity and vote metrics
Popularity, vote count, and vote average measure different things. A high-popularity movie may have strong audience engagement; it is not necessarily duplicated or erroneous. Likewise, a low rating can be plausible when supported by nearby ratings and a meaningful vote count.

In [6]:
metric_columns = [
    column
    for column in ("popularity", "vote_count", "vote_average")
    if column in investigation_movies.columns
]
for column in metric_columns:
    values = pd.to_numeric(investigation_movies[column], errors="coerce")
    print(
        f"{column}: missing={values.isna().sum()}, zero={values.eq(0).sum()}, "
        f"negative={values.lt(0).sum()}"
    )

if "popularity" in investigation_movies:
    print("Highest popularity candidates:")
    display(
        investigation_movies.sort_values("popularity", ascending=False)[
            ["movie_id", "title", "release_year", "popularity", "vote_count", "vote_average"]
        ].head(12).reset_index(drop=True)
    )

if "vote_average" in investigation_movies:
    print("Lowest vote averages:")
    display(
        investigation_movies.sort_values("vote_average", ascending=True)[
            ["movie_id", "title", "release_year", "vote_average", "vote_count", "popularity"]
        ].head(15).reset_index(drop=True)
    )

if "vote_count" in investigation_movies:
    print("Highest vote-count candidates:")
    display(
        investigation_movies.sort_values("vote_count", ascending=False)[
            ["movie_id", "title", "release_year", "vote_count", "popularity", "vote_average"]
        ].head(12).reset_index(drop=True)
    )

engagement_columns = [
    column
    for column in ("popularity", "vote_count", "vote_average", "budget", "revenue", "runtime")
    if column in investigation_movies.columns
]
print("Cross-field Pearson correlations:")
display(investigation_movies[engagement_columns].corr().round(3))

popularity: missing=0, zero=0, negative=0
vote_count: missing=0, zero=0, negative=0
vote_average: missing=0, zero=0, negative=0
Highest popularity candidates:


,movie_id,title,release_year,popularity,vote_count,vote_average
0,299534,Avengers: Endgame,2019,89.1177,29384,8.2470
1,155,The Dark Knight,2008,66.8926,37405,8.5400
2,157336,Interstellar,2014,56.3569,41925,8.4930
3,299536,Avengers: Infinity War,2018,49.4655,33418,8.2470
4,278,The Shawshank Redemption,1994,45.4678,31864,8.7340
5,27205,Inception,2010,45.4139,40754,8.4000
6,671,Harry Potter and the Philosopher's Stone,2001,43.3460,30644,7.9050
7,24428,The Avengers,2012,42.1051,40389,8.0850
8,238,The Godfather,1972,40.7425,23999,8.6880
9,120,The Lord of the Rings: The Fellowship of the Ring,2001,39.3976,28890,8.4510


Lowest vote averages:


,movie_id,title,release_year,vote_average,vote_count,popularity
0,41513,The Smurfs,2011,5.7520,4136,5.4347
1,217,Indiana Jones and the Kingdom of the Crystal S...,2008,6.0270,9386,7.9677
2,534,Terminator Salvation,2009,6.1080,7360,8.2923
3,955,Mission: Impossible II,2000,6.1400,7967,7.7691
4,8078,Alien Resurrection,1997,6.1700,5877,7.7192
5,331,Jurassic Park III,2001,6.1790,8195,0.4583
6,296,Terminator 3: Rise of the Machines,2003,6.2160,7641,9.4087
7,810,Shrek the Third,2007,6.3170,10138,11.1544
8,8077,Alien³,1992,6.4000,6722,7.7480
9,10192,Shrek Forever After,2010,6.4160,8237,13.7726


Highest vote-count candidates:


,movie_id,title,release_year,vote_count,popularity,vote_average
0,157336,Interstellar,2014,41925,56.3569,8.4930
1,27205,Inception,2010,40754,45.4139,8.4000
2,24428,The Avengers,2012,40389,42.1051,8.0850
3,155,The Dark Knight,2008,37405,66.8926,8.5400
4,299536,Avengers: Infinity War,2018,33418,49.4655,8.2470
5,550,Fight Club,1999,33362,33.3477,8.4000
6,278,The Shawshank Redemption,1994,31864,45.4678,8.7340
7,680,Pulp Fiction,1994,31350,29.9893,8.4800
8,13,Forrest Gump,1994,30907,28.9685,8.4620
9,671,Harry Potter and the Philosopher's Stone,2001,30644,43.3460,7.9050


Cross-field Pearson correlations:


,popularity,vote_count,vote_average,budget,revenue,runtime
popularity,1.0000,0.8300,0.5920,0.2630,0.5080,0.4840
vote_count,0.8300,1.0000,0.6590,0.2000,0.4070,0.4300
vote_average,0.5920,0.6590,1.0000,-0.3230,-0.0210,0.4180
budget,0.2630,0.2000,-0.3230,1.0000,0.7460,0.1990
revenue,0.5080,0.4070,-0.0210,0.7460,1.0000,0.2570
runtime,0.4840,0.4300,0.4180,0.1990,0.2570,1.0000


##### Preliminary review of the unusual fields
The following decisions are based on the current cache. They can be revisited if cache records are refreshed or a project-specific numeric rule is approved.

In [7]:
field_verdicts = [
    {
        "field": "budget",
        "flagged_movies": "Pirates of the Caribbean: On Stranger Tides; Avengers: Endgame",
        "evidence_of_normality": "Both values are positive and the films are known large-budget blockbusters; no negative or missing values occur.",
        "evidence_of_concern": "Nominal budgets are high and sit above the IQR upper bound.",
        "preliminary_conclusion": "Normal extreme",
    },
    {
        "field": "revenue",
        "flagged_movies": "Avengers: Endgame; Avengers: Infinity War",
        "evidence_of_normality": "Positive ratios and global blockbuster context support the high observations; no missing, zero, or negative values occur.",
        "evidence_of_concern": "Both values exceed the IQR upper bound and rank at the top of this cache.",
        "preliminary_conclusion": "Normal extreme",
    },
    {
        "field": "runtime",
        "flagged_movies": "The Godfather Part II; The Lord of the Rings: The Return of the King; Schindler's List",
        "evidence_of_normality": "All are positive, complete runtimes and are near other 195–202 minute films.",
        "evidence_of_concern": "Three runtimes exceed the IQR upper bound.",
        "preliminary_conclusion": "Normal extreme",
    },
    {
        "field": "popularity",
        "flagged_movies": "Avengers: Endgame; The Dark Knight; Interstellar; Avengers: Infinity War; Shawshank Redemption; Inception",
        "evidence_of_normality": "High values broadly align with high vote counts and no missing or negative values occur.",
        "evidence_of_concern": "Popularity is a cached/source-derived snapshot and can change independently of static film attributes.",
        "preliminary_conclusion": "Needs source verification",
    },
    {
        "field": "vote_average",
        "flagged_movies": "The Smurfs",
        "evidence_of_normality": "The 5.752 value is inside TMDB's 0–10 scale and is consistent with several nearby low-rated films.",
        "evidence_of_concern": "It is the sole IQR low tail value and has the smallest vote count among the cache.",
        "preliminary_conclusion": "Normal extreme",
    },
    {
        "field": "vote_count",
        "flagged_movies": "Interstellar; Inception",
        "evidence_of_normality": "Both are positive and high, with strong correlation to popularity and vote averages.",
        "evidence_of_concern": "Both exceed the IQR upper bound and are the cache's strongest vote-engagement observations.",
        "preliminary_conclusion": "Normal extreme",
    },
]
display(pd.DataFrame(field_verdicts))

,field,flagged_movies,evidence_of_normality,evidence_of_concern,preliminary_conclusion
0,budget,Pirates of the Caribbean: On Stranger Tides; A...,Both values are positive and the films are kno...,Nominal budgets are high and sit above the IQR...,Normal extreme
1,revenue,Avengers: Endgame; Avengers: Infinity War,Positive ratios and global blockbuster context...,Both values exceed the IQR upper bound and ran...,Normal extreme
2,runtime,The Godfather Part II; The Lord of the Rings: ...,"All are positive, complete runtimes and are ne...",Three runtimes exceed the IQR upper bound.,Normal extreme
3,popularity,Avengers: Endgame; The Dark Knight; Interstell...,High values broadly align with high vote count...,Popularity is a cached/source-derived snapshot...,Needs source verification
4,vote_average,The Smurfs,The 5.752 value is inside TMDB's 0–10 scale an...,It is the sole IQR low tail value and has the ...,Normal extreme
5,vote_count,Interstellar; Inception,"Both are positive and high, with strong correl...",Both exceed the IQR upper bound and are the ca...,Normal extreme


## 4. Release dates and common categories

Review coverage and the observed range before defining date rules. Frequency tables help identify dominant categories, spelling/label variants, or unexpected missing values.


In [8]:
if "release_date" in movies.columns:
    release_dates = movies["release_date"].dropna()
    print(f"Movies with a release date: {len(release_dates):,} / {len(movies):,}")
    if not release_dates.empty:
        print(f"Observed release-date range: {release_dates.min()} to {release_dates.max()}")
        release_year_counts = (
            release_dates.dt.year.value_counts().sort_index().rename("movie_count")
        )
        print("Earliest release years:")
        display(release_year_counts.head(15).to_frame())
        print("Latest release years:")
        display(release_year_counts.tail(15).to_frame())
else:
    print("No normalized release_date column is available.")

category_columns = [
    column
    for column in ("status", "original_language", "adult", "video")
    if column in movies.columns
]
category_rows = []
for column in category_columns:
    counts = movies[column].value_counts(dropna=False).head(15)
    for value, count in counts.items():
        category_rows.append(
            {
                "field": column,
                "value": value,
                "movie_count": int(count),
                "percent_of_movies": (
                    round(count / len(movies) * 100, 2) if len(movies) else pd.NA
                ),
            }
        )

category_profile = pd.DataFrame(category_rows)
display(category_profile)


Movies with a release date: 112 / 112
Observed release-date range: 1966-12-23 00:00:00 to 2019-06-19 00:00:00
Earliest release years:


,movie_count
release_date,
1966,1
1972,1
1974,1
1975,1
1977,1
1979,1
1980,1
1981,1
1982,1


Latest release years:


,movie_count
release_date,
2004,6
2005,3
2006,5
2007,6
2008,4
2009,3
2010,5
2011,4
2012,3


,field,value,movie_count,percent_of_movies
0,status,Released,112,100.0000
1,original_language,en,110,98.2100
2,original_language,ja,1,0.8900
3,original_language,it,1,0.8900
4,adult,False,112,100.0000
5,video,False,112,100.0000


## 5. Profile relationships

Compare relationship counts to all movies, including movies with zero links. Then inspect the most common genres. The left join preserves links even if a reference label is missing; the match count makes that case visible.


In [9]:
relationship_tables = {
    "genres": tables.movie_genres,
    "production companies": tables.movie_companies,
    "origin countries": tables.movie_origin_countries,
    "production countries": tables.movie_production_countries,
    "spoken languages": tables.movie_spoken_languages,
    "cast credits": tables.cast_credits,
    "crew credits": tables.crew_credits,
}

relationship_rows = []
for relationship_name, links in relationship_tables.items():
    counts_per_movie = links.groupby("movie_id").size().reindex(movies["movie_id"], fill_value=0)
    relationship_rows.append(
        {
            "relationship": relationship_name,
            "link_rows": len(links),
            "movies_with_at_least_one": int(counts_per_movie.gt(0).sum()),
            "movies_with_zero": int(counts_per_movie.eq(0).sum()),
            "median_links_per_movie_including_zero": (
                float(counts_per_movie.median()) if len(movies) else pd.NA
            ),
            "maximum_links_for_one_movie": (
                int(counts_per_movie.max()) if len(movies) else pd.NA
            ),
        }
    )

relationship_profile = pd.DataFrame(relationship_rows)
display(relationship_profile)

genre_counts = (
    tables.movie_genres.groupby("genre_id")
    .size()
    .rename("movie_count")
    .reset_index()
    .merge(
        tables.genres,
        on="genre_id",
        how="left",
        validate="many_to_one",
        indicator="genre_match",
    )
    .sort_values("movie_count", ascending=False)
)
display(genre_counts.head(20))
print("Genre-reference matches:")
display(genre_counts["genre_match"].value_counts(dropna=False).to_frame("link_rows"))


,relationship,link_rows,movies_with_at_least_one,movies_with_zero,median_links_per_movie_including_zero,maximum_links_for_one_movie
0,genres,320,112,0,3.0000,5
1,production companies,307,112,0,3.0000,7
2,origin countries,117,112,0,1.0000,2
3,production countries,154,112,0,1.0000,4
4,spoken languages,219,112,0,1.0000,6
5,cast credits,7827,112,0,65.5000,177
6,crew credits,24929,112,0,182.0000,953


,genre_id,movie_count,genre_name,genre_match
5,28,61,Action,both
0,12,60,Adventure,both
11,878,36,Science Fiction,both
9,53,33,Thriller,both
1,14,27,Fantasy,both
3,18,19,Drama,both
6,35,19,Comedy,both
10,80,18,Crime,both
2,16,17,Animation,both
14,10751,17,Family,both


Genre-reference matches:


,link_rows
genre_match,
both,16
left_only,0
right_only,0
